# LAB ASSIGNMENT 2 -- spotify

In [13]:
!pip install annoy faiss-cpu hnswlib

import pandas as pd
import numpy as np
import time
import faiss
import hnswlib
from annoy import AnnoyIndex
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/datasets/songs_with_attributes_and_lyrics.csv')
print(df.shape)
print(df.columns.tolist())
df.head()

(955320, 17)
['id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics']


,id,name,album_name,artists,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,lyrics
0,0Prct5TDjAnEgIqbxcldY9,!,UNDEN!ABLE,HELLYEAH,0.415,0.6050,7,-11.157,1,0.0575,0.00116,0.838000,0.4710,0.193,100.059,79500.0,"He said he came from Jamaica,\n he owned a cou..."
1,2ASl4wirkeYm3OWZxXKYuq,!!,NaN,Yxngxr1,0.788,0.6480,7,-9.135,0,0.3150,0.90000,0.000000,0.1760,0.287,79.998,114000.0,"Fucked a bitch, now she running with my kids\n..."
2,69lcggVPmOr9cvPx9kLiiN,!!! - Interlude,Where I Belong EP,Glowie,0.000,0.0354,7,-20.151,0,0.0000,0.90800,0.000000,0.4790,0.000,0.000,11413.0,"Oh, my God, I'm going crazy\n"
3,4U7dlZjg1s9pjdppqZy0fm,!!De Repente!!,Un Palo Al Agua (20 Grandes Canciones),Rosendo,0.657,0.8820,5,-6.340,1,0.0385,0.00740,0.000013,0.0474,0.939,123.588,198173.0,Continuamente se extraña la gente si no puede ...
4,4v1IBp3Y3rpkWmWzIlkYju,!!De Repente!!,Fuera De Lugar,Rosendo,0.659,0.8930,5,-8.531,1,0.0411,0.09220,0.000019,0.0534,0.951,123.600,199827.0,Continuamente se extraña la gente si no puede ...


In [15]:
num_df = df.select_dtypes(include='number').dropna()
print(num_df.columns.tolist())

# num_df = num_df.drop(columns=['Unnamed: 0'])  # contoh kalau ada kolom index

df = df.loc[num_df.index]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(num_df.values).astype('float32')

n_data, dim = X_scaled.shape
k = 10
print(n_data, dim)

['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms']
955320 10


In [16]:
np.random.seed(42)
q_idx = np.random.randint(0, n_data)
query = X_scaled[q_idx].reshape(1, -1)

name_col = 'name'
artists_col = 'artists'
print("Query song:", df.iloc[q_idx][name_col])

Query song: Caminho das Pedras


In [17]:
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
build_exact = time.time() - start

start = time.time()
_, idx_exact = nn.kneighbors(query)
time_exact = time.time() - start
idx_exact = idx_exact[0]

print("Exact NN:", idx_exact)

Exact NN: [121958 845348 443049 651777 722972 443048  12953 903724 287397 117008]


In [18]:
start = time.time()
annoy_index = AnnoyIndex(dim, 'euclidean')
for i in range(n_data):
    annoy_index.add_item(i, X_scaled[i])
annoy_index.build(10)
build_annoy = time.time() - start

start = time.time()
idx_annoy = annoy_index.get_nns_by_vector(query[0], k)
time_annoy = time.time() - start

print("ANNOY:", idx_annoy)

ANNOY: [121958, 845348, 443049, 722972, 443048, 12953, 474882, 57638, 201246, 845349]


In [19]:
start = time.time()
quantizer = faiss.IndexFlatL2(dim)
faiss_index = faiss.IndexIVFFlat(quantizer, dim, 100, faiss.METRIC_L2)
faiss_index.train(X_scaled)
faiss_index.add(X_scaled)
faiss_index.nprobe = 10
build_faiss = time.time() - start

start = time.time()
_, idx_faiss = faiss_index.search(query, k)
time_faiss = time.time() - start
idx_faiss = idx_faiss[0]

print("FAISS:", idx_faiss)

FAISS: [121958 845348 443049 651777 722972 443048  12953 903724 287397 117008]


In [20]:
start = time.time()
hnsw_index = hnswlib.Index(space='l2', dim=dim)
hnsw_index.init_index(max_elements=n_data, ef_construction=100, M=16)
hnsw_index.add_items(X_scaled)
hnsw_index.set_ef(50)
build_hnsw = time.time() - start

start = time.time()
idx_hnsw, _ = hnsw_index.knn_query(query, k=k)
time_hnsw = time.time() - start
idx_hnsw = idx_hnsw[0]

print("HNSW:", idx_hnsw)

HNSW: [121958 845348 443049 651777 722972 443048  12953 903724 287397 117008]


In [21]:
def recall(pred, gt):
  return len(set(pred) & set(gt)) / len(gt)

summary = pd.DataFrame({
  'Method': ['Exact NN', 'ANNOY', 'FAISS', 'HNSW'],
  'Build (s)': [build_exact, build_annoy, build_faiss, build_hnsw],
  'Query (ms)': [time_exact*1000, time_annoy*1000, time_faiss*1000, time_hnsw*1000],
  'Recall@10': [1.0,
                    recall(idx_annoy, idx_exact),
                    recall(idx_faiss, idx_exact),
                    recall(idx_hnsw, idx_exact)],
})
summary.round(4)

,Method,Build (s),Query (ms),Recall@10
0,Exact NN,0.0797,149.6258,1.0
1,ANNOY,27.5562,0.3068,0.6
2,FAISS,0.4659,6.1357,1.0
3,HNSW,116.6225,0.4094,1.0


In [22]:
cols = [name_col, artists_col]

for label, idx in [('Exact NN', idx_exact), ('ANNOY', idx_annoy),('FAISS', idx_faiss), ('HNSW', idx_hnsw)]:
  print(f'\n=== {label} ===')
  print(df.iloc[list(idx)][cols].to_string())


=== Exact NN ===
                                                                   name                      artists
121958                                               Caminho das Pedras   Quinteto em Branco e Preto
845348                                                  Up The Junction                      Squeeze
443049                                                         Le rital              Claude Barzotti
651777                                                     Rock with Me              Aretha Franklin
722972  Spirit in the Dark (with The Dixie Flyers) - 2018 Mono Remaster              Aretha Franklin
443048                                                         Le rital              Claude Barzotti
12953                                                       A Foggy Day    The George Benson Quartet
903724                   Ya Te Avisé - Remix JML Ruido Maldito Estudios  Bako, Fyahbwoy, Daddy Cobra
287397                                                   Get To Gettin'  